# P1 divergence census\n15 tasks x 5 runs, no memory, no hint, temperature 1.0. Executed decisions from the post-hoc extractor. Predictions: `quick_runs/predictions/P1.md`. Also: how the agent works (behaviour section).

In [1]:
import json, sys, statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd, yaml
REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "quick_runs"))
RUNS = Path("/data/anon/reveries_runs")
TASKS = {t["id"]: t for t in yaml.safe_load(open(REPO / "quick_runs/tasks.yaml"))["tasks"]}
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
from metrics import decision_agreement, first_divergence, answer_consistency, consistent_failures, score_spread, task_score
from reveries.decisions import DECISION_TYPES
from score import family_of, SF_CHECKS
DTYPES = [t for t in DECISION_TYPES if t != "final_selection"] + ["final_selection"]
def load_records(probe):
    recs = [json.loads(l) for l in open(RUNS / probe / "records.jsonl")]
    for r in recs:
        r["answer"] = tuple(r["answer"]) if isinstance(r["answer"], list) else r["answer"]
        r["risk"] = TASKS[r["task_id"]]["contamination_risk"]
        r["source"] = "SF" if r["task_id"].startswith("sf_") else "BLADE"
    return recs
def by_task(recs):
    d = defaultdict(list)
    for r in recs: d[r["task_id"]].append(r)
    return dict(sorted(d.items()))
def task_table(groups):
    rows = []
    for t, rs in groups.items():
        row = {"task": t, "risk": TASKS[t]["contamination_risk"], "n": len(rs),
               "answer_consistency": answer_consistency(rs), "score_mean": task_score(rs), "score_sd": score_spread(rs),
               "first_divergence": first_divergence(rs, DTYPES),
               "consistent_failures": ",".join(consistent_failures(rs))}
        for dt in DTYPES:
            row[dt] = decision_agreement(rs, dt)
        rows.append(row)
    return pd.DataFrame(rows)

In [3]:
recs = load_records("P1"); groups = by_task(recs)
print(len(recs), "runs;", Counter(r["stop"] for r in recs))
tt = task_table(groups)
tt[["task", "risk", "n", "answer_consistency", "score_mean", "score_sd", "first_divergence", "consistent_failures"]]

75 runs; Counter({'final_answer': 75})


,task,risk,n,answer_consistency,score_mean,score_sd,first_divergence,consistent_failures
0,blade_boxes,medium,5,1.0,1.000000e+00,0.000000,framing.target,
1,blade_caschools,high,5,0.6,1.000000e+00,0.000000,framing.target,
2,blade_crofoot,high,5,1.0,1.000000e+00,0.000000,variables,
3,blade_fish,high,5,1.0,7.333333e-01,0.149071,framing.population,
4,blade_hurricane,high,5,1.0,1.000000e+00,0.000000,variables,
5,blade_reading,medium,5,0.4,6.666667e-01,0.000000,framing.target,dv_matches_expert
6,blade_teachingratings,high,5,1.0,1.000000e+00,0.000000,framing.unit,
7,sf_tf01_goal_imbalance,high,5,0.1,7.573423e-01,0.062846,variables,"checks_class_distribution,uses_class_weight,uses_resampling"
8,sf_tf02_player_rep_norm,low,5,1.0,4.664201e+06,0.000000,transformation,handles_unit_suffix_in_height
9,sf_tf03_correlation_causation,low,5,1.0,2.769464e+07,3.067480,variables,"applies_causal_reasoning,column_x_included"


In [4]:
# Decision agreement per type, per task (share of run pairs with the same executed value)
tt.set_index("task")[DTYPES].round(2)

,framing.target,framing.population,framing.unit,variables,transformation,model.family,model.controls,validation,metric,verdict,final_selection
task,,,,,,,,,,,
blade_boxes,0.1,0.3,0.6,0.1,0.0,0.6,0.17,1.0,1.0,1.0,0.0
blade_caschools,0.1,1.0,0.6,1.0,0.1,1.0,0.10,1.0,1.0,0.6,0.0
blade_crofoot,1.0,1.0,1.0,0.0,0.0,1.0,NaN,1.0,1.0,1.0,0.0
blade_fish,1.0,0.6,0.6,0.2,0.1,0.6,0.17,1.0,1.0,1.0,0.0
blade_hurricane,1.0,1.0,1.0,0.6,1.0,1.0,0.60,1.0,1.0,1.0,0.0
blade_reading,0.6,0.6,0.3,1.0,0.3,1.0,0.60,1.0,1.0,1.0,0.0
blade_teachingratings,1.0,1.0,0.6,1.0,NaN,0.6,0.10,1.0,1.0,1.0,0.1
sf_tf01_goal_imbalance,1.0,1.0,1.0,0.6,0.4,1.0,NaN,0.6,0.3,NaN,1.0
sf_tf02_player_rep_norm,1.0,1.0,1.0,1.0,0.6,1.0,NaN,1.0,1.0,NaN,1.0


In [5]:
# Decision types ranked by disagreement (pooled = mean over tasks), split by source
rank = pd.DataFrame({"all": tt[DTYPES].mean(), "SF": tt[tt.task.str.startswith("sf_")][DTYPES].mean(),
                     "BLADE": tt[tt.task.str.startswith("blade_")][DTYPES].mean(),
                     "tasks_all_same": (tt[DTYPES] == 1.0).sum()}).sort_values("all")
rank.round(2)

,all,SF,BLADE,tasks_all_same
transformation,0.27,0.29,0.25,1
model.controls,0.29,NaN,0.29,0
final_selection,0.35,0.64,0.01,4
variables,0.52,0.49,0.56,5
framing.unit,0.73,0.79,0.67,7
model.family,0.83,0.82,0.83,10
framing.target,0.85,1.00,0.69,12
framing.population,0.90,1.00,0.79,12
verdict,0.94,NaN,0.94,6
metric,0.95,0.91,1.00,14


In [6]:
# First divergence: how often is it one of the first two decision types the runs state?
def first_two(rs):
    c = Counter(t for r in rs for t in r["order"][:2]); return {t for t, _ in c.most_common(2)}
fd = pd.DataFrame([{"task": t, "first_divergence": first_divergence(rs, DTYPES), "first_two_stated": sorted(first_two(rs))} for t, rs in groups.items()])
fd["in_first_two"] = [f in s for f, s in zip(fd.first_divergence, fd.first_two_stated)]
print(Counter(fd.first_divergence)); fd

Counter({'variables': 7, 'framing.target': 3, 'framing.unit': 2, 'transformation': 2, 'framing.population': 1})


,task,first_divergence,first_two_stated,in_first_two
0,blade_boxes,framing.target,"[framing.population, framing.target]",True
1,blade_caschools,framing.target,"[framing.population, framing.target]",True
2,blade_crofoot,variables,"[framing.population, framing.target]",False
3,blade_fish,framing.population,"[framing.population, framing.target]",True
4,blade_hurricane,variables,"[framing.population, framing.target]",False
5,blade_reading,framing.target,"[framing.population, framing.target]",True
6,blade_teachingratings,framing.unit,"[framing.population, framing.target]",False
7,sf_tf01_goal_imbalance,variables,"[framing.population, framing.target]",False
8,sf_tf02_player_rep_norm,transformation,"[framing.population, framing.target]",False
9,sf_tf03_correlation_causation,variables,"[framing.population, framing.target]",False


In [7]:
# Fork list: task, decision type, the options, runs per option (types with > 1 executed value)
forks = []
for t, rs in groups.items():
    for dt in DTYPES:
        c = Counter(r["decisions"].get(dt) for r in rs if r["decisions"].get(dt) is not None)
        if len(c) > 1:
            forks.append({"task": t, "type": dt, "n_options": len(c), "options": dict(c.most_common())})
forks = pd.DataFrame(forks)
print(len(forks), "forks over", forks.task.nunique(), "tasks")
forks[forks.type.isin(["variables", "model.family", "model.controls", "validation", "metric", "verdict", "framing.population"])]

62 forks over 15 tasks


,task,type,n_options,options
1,blade_boxes,framing.population,3,"{'all rows': 3, 'age < 14': 1, 'y in [2, 3]': 1}"
3,blade_boxes,variables,4,"{'age': 2, 'age|culture|i(age**2)': 1, 'age|age_sq': 1, 'age|culture|gender|..."
5,blade_boxes,model.family,2,"{'logistic': 4, 'multinomial_logistic': 1}"
6,blade_boxes,model.controls,3,"{'culture|gender|majority_first': 2, 'gender|majority_first': 1, 'culture|ma..."
11,blade_caschools,model.controls,4,"{'english|income|lunch': 2, 'english|expenditure|income|lunch': 1, 'calworks..."
12,blade_caschools,verdict,2,"{'supported': 4, 'rejected': 1}"
14,blade_crofoot,variables,5,"{'home_advantage|relative_male_size': 1, 'dist_diff|size_diff': 1, 'rel_dist..."
17,blade_fish,framing.population,2,"{'all rows': 4, 'hours > 0': 1}"
19,blade_fish,variables,3,"{'livebait': 2, 'camper|child|livebait|persons': 2, 'camper|child|livebait': 1}"
21,blade_fish,model.family,2,"{'negative_binomial': 4, 'poisson': 1}"


In [8]:
# Consistent failures and per-check pass rates (Silent Failures: judge majority of 3 draws; BLADE: expert match)
cf = []
for t, rs in groups.items():
    for c in sorted(set().union(*(r["checks"] for r in rs))):
        v = [r["checks"].get(c) for r in rs]
        cf.append({"task": t, "check": c, "pass": v.count(True), "fail": v.count(False), "unknown": v.count(None),
                   "consistent_failure": all(x is False for x in v)})
cf = pd.DataFrame(cf); cf

,task,check,pass,fail,unknown,consistent_failure
0,blade_boxes,avoids_expert_negative,5,0,0,False
1,blade_boxes,dv_matches_expert,5,0,0,False
2,blade_boxes,family_matches_expert,5,0,0,False
3,blade_boxes,iv_matches_expert,5,0,0,False
4,blade_caschools,avoids_expert_negative,5,0,0,False
5,blade_caschools,dv_matches_expert,5,0,0,False
6,blade_caschools,family_matches_expert,5,0,0,False
7,blade_caschools,iv_matches_expert,5,0,0,False
8,blade_crofoot,avoids_expert_negative,5,0,0,False
9,blade_crofoot,dv_matches_expert,5,0,0,False


In [9]:
# Judge stability (Silent Failures): share of (run, check) where the 3 judge draws agree
ja = [a for r in recs if r["source"] == "SF" for a in r["judge_agreement"].values()]
print("judge draws agree on", sum(ja), "of", len(ja), "run-checks")

judge draws agree on 79 of 80 run-checks


In [10]:
# Behaviour: how the agent works
def run_events(r):
    return [json.loads(l) for l in open(RUNS / "P1" / r["task_id"] / r["run"] / "events.jsonl")]
beh = []
for r in recs:
    ev = run_events(r)
    cells = [e for e in ev if e["event"] == "code_executed"]
    stated_fam = [family_of(e["value"]) for e in ev if e["event"] == "decision_stated" and e["type"] == "model.family"]
    ex_fam = r["executed"].get("model.family")
    beh.append({"task": r["task_id"], "source": r["source"], "run": r["run"], "stop": r["stop"], "steps": r["steps"],
        "code_cells": len(cells), "cell_errors": sum(not c["ok"] for c in cells),
        "no_code_turns": sum(e["event"] == "no_code" for e in ev),
        "stated_decisions": len(r["stated"]), "stated_types": len(set(d["type"] for d in r["stated"])),
        "silent_types": len(r["executed"].get("silent") or []),
        "families_fitted": len(r["executed"].get("model.all_families") or []),
        "stated_family_last": stated_fam[-1] if stated_fam else None, "executed_family": ex_fam,
        "family_mismatch": (bool(stated_fam) and ex_fam not in (None, "other") and stated_fam[-1] not in ("other", "several")
                            and {"linear_regression": "ols"}.get(ex_fam, ex_fam) != stated_fam[-1]),
        "completion_tokens": r["tokens"]["completion"], "reasoning_tokens": r["tokens"]["reasoning"],
        "minutes": (ev[-1]["t"] - ev[0]["t"]) / 60})
beh = pd.DataFrame(beh)
beh.groupby("source")[["steps", "code_cells", "cell_errors", "no_code_turns", "stated_decisions", "silent_types",
                      "families_fitted", "completion_tokens", "reasoning_tokens", "minutes"]].describe().T.round(1)

source         BLADE    SF
steps   count   35.0  40.0
        mean     6.1   3.2
        std      1.6   1.1
        min      3.0   2.0
        25%      5.0   2.0
...              ...   ...
minutes min      7.7   1.7
        25%     12.4   3.0
        50%     17.8   3.6
        75%     21.4   4.2
        max     26.9  15.3

[80 rows x 2 columns]

In [11]:
# Stated vs executed model family (keyword map of the last stated model.family vs the extractor)
beh[beh.family_mismatch][["task", "run", "stated_family_last", "executed_family"]]

,task,run,stated_family_last,executed_family


In [12]:
# Which decision types are made silently (made in code, never announced)?
sil = Counter(t for r in recs for t in (r["executed"].get("silent") or []))
pd.Series(sil).sort_values(ascending=False)

validation            28
framing.unit          27
framing.population    26
transformation        25
metric                18
model.all_families    11
model.controls        10
model.family           8
framing.target         7
variables              7
final_selection        6
dtype: int64

In [13]:
# Results split by contamination risk (after P10)
tt.groupby("risk")[["answer_consistency", "score_sd"] + DTYPES].mean().round(2).T

risk,high,low,medium
answer_consistency,0.76,1.00,0.88
score_sd,0.03,1.53,0.00
framing.target,0.89,1.00,0.74
framing.population,0.95,1.00,0.78
framing.unit,0.85,1.00,0.44
variables,0.57,0.65,0.38
transformation,0.31,0.35,0.18
model.family,0.82,1.00,0.76
model.controls,0.24,NaN,0.38
validation,0.95,1.00,1.00


In [14]:
summary = {"runs": len(recs), "stops": dict(Counter(r["stop"] for r in recs)),
  "agreement_rank_all": rank["all"].round(3).to_dict(), "agreement_SF": rank["SF"].round(3).to_dict(),
  "agreement_BLADE": rank["BLADE"].round(3).to_dict(), "tasks_all_same_by_type": rank["tasks_all_same"].to_dict(),
  "first_divergence_counts": dict(Counter(fd.first_divergence)), "first_divergence_in_first_two": int(fd.in_first_two.sum()),
  "n_forks": len(forks), "forks": forks.to_dict("records"),
  "consistent_failures": cf[cf.consistent_failure][["task", "check"]].to_dict("records"),
  "check_table": cf.to_dict("records"),
  "judge_draws_agree": [int(sum(ja)), len(ja)],
  "answer_consistency_by_source": tt.assign(src=tt.task.str.split("_").str[0]).groupby("src").answer_consistency.mean().round(3).to_dict(),
  "per_task": tt.set_index("task").to_dict("index"),
  "behaviour_by_source": beh.groupby("source")[["steps", "code_cells", "cell_errors", "stated_decisions", "silent_types",
       "families_fitted", "completion_tokens", "minutes"]].mean().round(2).to_dict("index"),
  "family_mismatch_runs": int(beh.family_mismatch.sum()), "silent_type_counts": dict(sil),
  "by_risk": tt.groupby("risk")[["answer_consistency", "score_sd"]].mean().round(3).to_dict("index")}
(REPO / "quick_runs/results/P1").mkdir(parents=True, exist_ok=True)
json.dump(summary, open(REPO / "quick_runs/results/P1/summary.json", "w"), indent=1, default=str)
forks.to_csv(REPO / "quick_runs/results/P1/forks.csv", index=False); cf.to_csv(REPO / "quick_runs/results/P1/checks.csv", index=False)
beh.to_csv(REPO / "quick_runs/results/P1/behaviour.csv", index=False)
"done" 

'done'